# Spark DataFrame Optimization

A runnable lab for writing efficient DataFrame queries, observing how Catalyst and Adaptive Query Execution (AQE) optimize them, and deciding when physical files are useful.

> Run the cells in order. The examples use generated data, so the lab has no external data dependency. Spark 3.5+ is recommended.

## Learning goals

By the end you can:

- separate logical optimization from physical execution optimization;
- verify an optimization with `explain`, metrics, and the Spark UI;
- reduce scan I/O with column pruning, predicate pushdown, and partition pruning;
- control joins, shuffles, skew, aggregation, caching, and partition counts;
- stage Parquet data locally or in HDFS and avoid the small-files problem.

## 0. Setup and reproducibility

The configuration values below must be set before the first action. AQE lets Spark change parts of the physical plan after it receives runtime statistics. The local master is only created when the notebook is not already attached to a Spark cluster.

In [ ]:
from pathlib import Path
import tempfile
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.storagelevel import StorageLevel

builder = (SparkSession.builder
           .appName("Spark-DataFrame-Optimization")
           .config("spark.sql.adaptive.enabled", "true")
           .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
           .config("spark.sql.adaptive.skewJoin.enabled", "true")
           .config("spark.sql.shuffle.partitions", "16"))
if SparkSession.getActiveSession() is None:
    builder = builder.master("local[*]")
spark = builder.getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(spark.version)
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))

## 1. Cleanly stage reusable demonstration data

Writing files is worthwhile here because scan optimizations only appear on a file source. It also makes repeated experiments deterministic and separates data generation from query timing. Parquet supplies columnar storage, embedded schema, per-file/row-group statistics, compression, and predicate pushdown.

`STAGE_ROOT` defaults to a new temporary directory. On a Hadoop cluster replace it with, for example, `hdfs:///tmp/dataeng/spark_optimization`. Do not use `Path(...)` for HDFS URIs; Spark accepts the URI directly. The overwrite is limited to this dedicated lab directory.

In [ ]:
# Safe default. For HDFS use: STAGE_ROOT = "hdfs:///tmp/dataeng/spark_optimization"
STAGE_ROOT = tempfile.mkdtemp(prefix="spark_optimization_")
sales_path = f"{STAGE_ROOT}/sales_parquet"
customers_path = f"{STAGE_ROOT}/customers_parquet"

customers = (spark.range(0, 2_000)
    .select(F.col("id").alias("customer_id"),
            F.concat(F.lit("customer_"), F.col("id")).alias("customer_name"),
            F.element_at(F.array(*[F.lit(x) for x in ["IN", "US", "GB", "DE"]]),
                         (F.col("id") % 4 + 1).cast("int")).alias("country"),
            F.when(F.col("id") < 40, "VIP").otherwise("STANDARD").alias("segment")))

sales = (spark.range(0, 200_000, numPartitions=16)
    .select(F.col("id").alias("sale_id"),
            (F.col("id") % 2_000).alias("customer_id"),
            F.date_add(F.lit("2025-01-01").cast("date"), (F.col("id") % 365).cast("int")).alias("sale_date"),
            F.round((F.col("id") % 500) * 1.17 + 5, 2).alias("amount"),
            F.element_at(F.array(*[F.lit(x) for x in ["web", "store", "partner"]]),
                         (F.col("id") % 3 + 1).cast("int")).alias("channel"),
            (F.col("id") % 25).cast("int").alias("product_id"))
    .withColumn("sale_year", F.year("sale_date"))
    .withColumn("sale_month", F.month("sale_date")))

(sales.write.mode("overwrite")
 .partitionBy("sale_year", "sale_month")
 .parquet(sales_path))
customers.write.mode("overwrite").parquet(customers_path)

sales_disk = spark.read.parquet(sales_path)
customers_disk = spark.read.parquet(customers_path)
print(STAGE_ROOT, sales_disk.count(), customers_disk.count())

## A repeatable way to inspect optimization

`extended` shows parsed, analyzed, optimized logical, and physical plans. `formatted` is easier for checking scan details such as `ReadSchema`, `PushedFilters`, and `PartitionFilters`. Plans are evidence of intent; the SQL tab, stage metrics, input bytes, shuffle bytes, task duration, and spill in the Spark UI are evidence of runtime behavior.

In [ ]:
def inspect(df, mode="formatted", run=False, rows=5):
    df.explain(mode=mode)
    if run:
        df.show(rows, truncate=False)

# Use inspect(query), then inspect(query, run=True) when execution is desired.

# 2. Projection pruning

### Optimization name
Column pruning (projection pushdown).

### How to write the query
Select only the columns required downstream. Avoid `select('*')` in production pipelines, especially before joins.

### How Spark optimizes it
Catalyst propagates required attributes toward the scan. A columnar Parquet reader then reads only those physical columns. In the formatted plan, `ReadSchema` should contain `customer_id`, `amount`, and partition columns—not every sales column.

### Deeper topic
A narrow schema reduces disk/network I/O, decoded vectors, memory pressure, serialization, and shuffle width. Projection pruning is automatic only when Spark can see through the expression; opaque Python UDFs and converting to RDDs can hide information from Catalyst.

In [ ]:
projection_query = (sales_disk
    .select("customer_id", "amount")
    .where(F.col("amount") >= 400))
inspect(projection_query)

# 3. Predicate pushdown and filter simplification

### Optimization name
Predicate pushdown, constant folding, and Boolean simplification.

### How to write the query
Use built-in column expressions and filter early for readability. Compare columns with literals of a compatible type. Prefer range predicates over wrapping a scanned column in a function.

### How Spark optimizes it
Catalyst combines filters, folds constants, and pushes eligible predicates to the data source. Parquet statistics can skip row groups/files. Look for `PushedFilters: [IsNotNull(amount), GreaterThanOrEqual(amount,400.0)]`. The remaining Spark `Filter` is still needed because source filters may not be exact.

### Deeper topic
`year(sale_date) = 2025` is generally less pushdown-friendly than a half-open range. Python UDF predicates are black boxes to Catalyst and add Python/JVM serialization. Built-in Spark functions preserve optimizer visibility and code generation.

In [ ]:
pushdown_query = (sales_disk
    .where((F.col("amount") >= 400.0) &
           (F.col("sale_date") >= F.lit("2025-06-01").cast("date")) &
           (F.col("sale_date") < F.lit("2025-07-01").cast("date")))
    .select("sale_id", "sale_date", "amount"))
inspect(pushdown_query)

# 4. Partition pruning

### Optimization name
Static partition pruning.

### How to write the query
Filter directly on the directory partition columns (`sale_year`, `sale_month`). Include a data-column date range too when it is part of the business rule.

### How Spark optimizes it
Spark uses partition directory metadata to avoid opening unrelated paths. `PartitionFilters` in the scan should contain year and month, while `PushedFilters` applies to columns stored inside Parquet. These are different mechanisms and can cooperate.

### Deeper topic
Partitioning is a physical data-layout decision, not the same as `repartition()` during computation. Choose low/medium-cardinality columns commonly filtered on. Too many partition values create tiny directories/files and expensive metadata listing. Dynamic partition pruning can also use join-side values at runtime for partitioned fact tables.

In [ ]:
partition_query = (sales_disk
    .where((F.col("sale_year") == 2025) & (F.col("sale_month") == 6))
    .groupBy("channel")
    .agg(F.sum("amount").alias("revenue")))
inspect(partition_query)

# 5. Join selection and broadcasting

### Optimization name
Broadcast hash join (BHJ).

### How to write the query
Filter and project the dimension first. Let Spark choose from statistics, or use `F.broadcast()` only when you know the relation safely fits in executor memory.

### How Spark optimizes it
The small side is collected, broadcast once to executors, and placed in an in-memory hash table. The large side is streamed without a two-sided shuffle. Look for `BroadcastExchange` and `BroadcastHashJoin`.

### Deeper topic
A broadcast hint overrides cost-based preference but not physical feasibility. Broadcasting a relation that expands unexpectedly can cause driver/executor memory failures and timeout. Without reliable statistics, Spark may choose poorly; persisted catalog tables benefit from computed statistics. AQE can convert some sort-merge joins to broadcast joins once runtime size is known.

In [ ]:
small_dimension = customers_disk.select("customer_id", "customer_name", "segment")
join_query = (sales_disk
    .where(F.col("sale_month") == 6)
    .select("customer_id", "amount")
    .join(F.broadcast(small_dimension), "customer_id", "inner")
    .groupBy("segment")
    .agg(F.sum("amount").alias("revenue")))
inspect(join_query)

# 6. Shuffle control: repartition, coalesce, and partition count

### Optimization name
Intentional shuffle partitioning.

### How to write the query
Use `repartition(n, key)` before repeated wide operations on the same key, or to distribute output by key. Use `coalesce(n)` mainly to reduce partitions after a large filter without a full reshuffle. Do not sprinkle either call into every pipeline.

### How Spark optimizes it
Wide operations introduce `Exchange` nodes. `repartition` explicitly creates a shuffle; `coalesce` usually collapses existing partitions without balancing them. AQE can combine small post-shuffle partitions according to advisory size.

### Deeper topic
Too few partitions underuse cores and create large spill-prone tasks; too many create scheduling and tiny-block overhead. Tune from observed shuffle bytes and target reasonably sized tasks, not a universal magic number. `spark.sql.shuffle.partitions` is an initial value under AQE, not necessarily the final count.

In [ ]:
by_customer = (sales_disk
    .select("customer_id", "amount")
    .repartition(16, "customer_id")
    .groupBy("customer_id")
    .agg(F.sum("amount").alias("lifetime_value")))
print("Declared output partitions:", by_customer.rdd.getNumPartitions())
inspect(by_customer)

# Reduction after a selective filter; inspect balance before using this in production.
small_result = sales_disk.where(F.col("amount") > 580).coalesce(2)

# 7. Efficient aggregation

### Optimization name
Partial (map-side) aggregation and narrow inputs.

### How to write the query
Filter and select before `groupBy`; use built-in associative aggregates such as `sum`, `count`, `min`, and `max`. Aggregate before a join when the business semantics allow it.

### How Spark optimizes it
Spark commonly emits a partial `HashAggregate` before the `Exchange` and a final aggregate after it. Local combination reduces the number and width of records crossing the network.

### Deeper topic
`groupByKey`-style logic retains all values per key and is often much heavier than declarative aggregates. Exact `countDistinct` can be expensive; `approx_count_distinct` uses a bounded-memory sketch when approximation is acceptable. Inspect spill and peak execution memory for high-cardinality groups.

In [ ]:
aggregation_query = (sales_disk
    .where(F.col("sale_month").between(1, 3))
    .select("product_id", "amount")
    .groupBy("product_id")
    .agg(F.sum("amount").alias("revenue"),
         F.count(F.lit(1)).alias("sales_count"),
         F.approx_count_distinct("amount").alias("approx_price_points")))
inspect(aggregation_query)

# 8. Data skew and salting

### Optimization name
Skew diagnosis, AQE skew handling, and manual salting.

### How to write the query
Measure key frequencies first. Keep AQE skew join enabled. Salt only confirmed hot keys; replicate the matching dimension rows across the same salt range. For aggregations, salt first and aggregate again by the original key.

### How Spark optimizes it
AQE observes shuffle partition sizes and can split skewed sort-merge-join partitions and replicate their matching side. Manual salting changes the key distribution so several tasks share one hot key.

### Deeper topic
Skew means one or a few tasks take far longer than the median. More partitions alone does not split identical keys. Salting adds complexity and may expand the other join side, so validate it with task-duration and shuffle-read distributions. The code below demonstrates aggregation salting without requiring a join.

In [ ]:
skewed = (spark.range(0, 100_000, numPartitions=8)
    .select(F.when(F.col("id") < 80_000, F.lit(0))
              .otherwise(F.col("id") % 100).alias("key"),
            F.lit(1).alias("value")))
skewed.groupBy("key").count().orderBy(F.desc("count")).show(5)

salt_buckets = 8
salted_partial = (skewed
    .withColumn("salt", F.pmod(F.xxhash64(F.monotonically_increasing_id()), F.lit(salt_buckets)))
    .groupBy("key", "salt").agg(F.sum("value").alias("partial_total")))
salted_result = salted_partial.groupBy("key").agg(F.sum("partial_total").alias("total"))
inspect(salted_result)

# 9. Cache and persist only reused work

### Optimization name
Materialization with cache/persist.

### How to write the query
Persist an expensive deterministic DataFrame only when it will be reused by multiple actions. Trigger one materializing action, reuse it, and `unpersist()` promptly.

### How Spark optimizes it
After materialization, later queries can read an in-memory relation instead of replaying lineage. Spark remains lazy: calling `persist` alone does not compute anything.

### Deeper topic
Caching a one-use DataFrame makes the job slower, and caching raw wide input wastes memory. Choose a storage level based on recomputation cost and memory. Cache invalidation, executor loss, and eviction mean a cache is an optimization—not durable storage or a correctness dependency.

In [ ]:
reused = (sales_disk
    .where(F.col("sale_year") == 2025)
    .select("customer_id", "product_id", "amount")
    .persist(StorageLevel.MEMORY_AND_DISK))
reused.count()  # materialize once
reused.groupBy("product_id").agg(F.sum("amount")).show(5)
reused.groupBy("customer_id").agg(F.avg("amount")).show(5)
reused.unpersist()

# 10. Small files, output sizing, and compaction

### Optimization name
Write-path partition sizing and compaction.

### How to write the query
Estimate output size, choose a reasonable partition count, repartition by physical partition columns when useful, and then write. `maxRecordsPerFile` places an upper bound on records per file; it does not combine small partitions.

### How Spark optimizes it
Spark generally writes one file per output task per dynamic partition touched. It does not automatically compact arbitrary historical files in plain Parquet. AQE may coalesce shuffle partitions before the writer, but deliberate table maintenance is still needed.

### Deeper topic
Tiny files increase listing, scheduling, open, and metadata costs. One huge file destroys parallelism. Target sizes depend on storage and workload; measure compressed bytes rather than relying only on row counts. `coalesce(1)` is suitable only for genuinely tiny exports because it serializes the write through one task. Table formats such as Delta, Iceberg, or Hudi may provide dedicated compaction operations.

In [ ]:
optimized_output_path = f"{STAGE_ROOT}/sales_summary"
summary = (sales_disk
    .groupBy("sale_year", "sale_month", "channel")
    .agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders")))

(summary.repartition(4, "sale_year", "sale_month")
 .write.mode("overwrite")
 .option("compression", "snappy")
 .option("maxRecordsPerFile", 100_000)
 .partitionBy("sale_year", "sale_month")
 .parquet(optimized_output_path))
spark.read.parquet(optimized_output_path).show(10, truncate=False)

# 11. Built-in expressions, UDFs, and whole-stage code generation

### Optimization name
Catalyst-visible expressions and whole-stage code generation.

### How to write the query
Prefer DataFrame functions (`when`, arithmetic, date/string functions, higher-order functions) over Python UDFs. Use a UDF only when the logic cannot be represented reasonably with built-ins.

### How Spark optimizes it
Catalyst can fold, reorder, prune, and generate JVM bytecode for built-in expressions. Compatible physical operators are fused into codegen stages, reducing virtual calls and intermediate rows. A normal Python UDF introduces a Python execution boundary.

### Deeper topic
Pandas UDFs use Arrow batches and can be much better than row-at-a-time Python UDFs, but they still limit optimizer visibility and cross a language boundary. Verify code generation with `explain(mode='codegen')`; not every operator participates.

In [ ]:
built_in_query = (sales_disk
    .select("sale_id",
            F.when(F.col("amount") >= 400, F.lit("HIGH"))
             .when(F.col("amount") >= 150, F.lit("MEDIUM"))
             .otherwise(F.lit("LOW")).alias("value_band")))
built_in_query.explain(mode="codegen")

# 12. AQE: let runtime evidence refine the plan

### Optimization name
Adaptive Query Execution.

### How to write the query
Write declarative DataFrame logic and keep AQE enabled. Avoid unnecessary hints that lock Spark into a strategy before runtime sizes are available. Execute an action before inspecting the final adaptive plan.

### How Spark optimizes it
AQE uses materialized shuffle statistics to coalesce small partitions, change some sort-merge joins into broadcast/shuffled-hash joins, and split skewed join partitions. Before an action, `AdaptiveSparkPlan isFinalPlan=false` is expected.

### Deeper topic
AQE does not remove the need for good layout, selective predicates, or sufficient cluster memory. Its decisions occur at query-stage boundaries. Compare the initial and final plans after an action, and correlate them with UI metrics.

In [ ]:
aqe_query = (sales_disk
    .join(customers_disk, "customer_id")
    .groupBy("country", "sale_month")
    .agg(F.sum("amount").alias("revenue")))
print("Before action")
aqe_query.explain(mode="formatted")
aqe_query.collect()
print("After action — inspect isFinalPlan and QueryStage nodes")
aqe_query.explain(mode="formatted")

# 13. End-to-end optimized DataFrame pattern

This final query combines physical partition pruning, data-source pushdown, projection pruning, pre-aggregation, and a small broadcast join. Optimization is compositional: the largest gains usually come from moving fewer bytes through the entire plan.

In [ ]:
fact_reduced = (sales_disk
    .where((F.col("sale_year") == 2025) &
           (F.col("sale_month").between(4, 6)) &
           (F.col("amount") >= 100))
    .select("customer_id", "amount")
    .groupBy("customer_id")
    .agg(F.sum("amount").alias("customer_revenue")))

final_query = (fact_reduced
    .join(F.broadcast(customers_disk.select("customer_id", "country", "segment")),
          "customer_id")
    .groupBy("country", "segment")
    .agg(F.sum("customer_revenue").alias("revenue"))
    .orderBy(F.desc("revenue")))
inspect(final_query, run=True, rows=20)

# Optimization review checklist

Before changing configuration, ask:

1. **Scan:** Are `ReadSchema`, `PushedFilters`, and `PartitionFilters` as selective as expected?
2. **Join:** Is the build side genuinely small? Are statistics reliable? Is skew visible?
3. **Shuffle:** Which `Exchange` nodes are necessary? Are task sizes balanced? Is there spill?
4. **Aggregation:** Does partial aggregation reduce data before the exchange?
5. **Reuse:** Is an expensive result used enough times to justify persistence?
6. **Files:** Are output files neither tiny nor too large, and is the partition layout aligned with filters?
7. **Evidence:** Did elapsed time, input bytes, shuffle bytes, spill, and task distribution improve over a baseline?

Change one factor at a time and preserve the same data and action. A shorter plan is not automatically a faster plan, and a faster first run may only reflect cache or warm filesystem effects.

## Optional cleanup

The local temporary stage is intentionally not deleted automatically so you can inspect its Parquet layout. Remove it after the lab. For HDFS use `hdfs dfs -rm -r /tmp/dataeng/spark_optimization` only after confirming the exact path.

In [ ]:
# Local cleanup (uncomment only after checking STAGE_ROOT):
# import shutil
# assert Path(STAGE_ROOT).name.startswith("spark_optimization_")
# shutil.rmtree(STAGE_ROOT)

# spark.stop()  # Uncomment when the notebook owns the Spark session.